# Portfolio Manager

Clean pipeline for mutual fund portfolio construction.

**Pipeline:**
1. Environment Setup
2. Paths & Parameters
3. Utility Functions
4. Core Logic (Screen → Rank → Overlap → Optimize)
5. Load & Validate Data
6. Resolve Fund Names
7. Run Pipeline
8. Results

**Design principles:**
- Input fund list is already shortlisted — no fund is hard-dropped. Missing data scores neutral (50 pts) for that component, not disqualified.
- Hybrid score = `abs_score` (pre-tested, upstream) + `risk_score` (peer-relative, computed here).
- `risk_score` components: Sharpe 3Y (25%) + Sortino 3Y (20%) + Sharpe 5Y (25%) + Expense Ratio (20%) + AUM penalty for small/mid cap (10%).
- Both components normalized to [0, 100] within the universe before blending — so the 50/50 split is genuine.
- Overlap uses Sørensen (average) normalization, not min.
- Optimizer relaxes overlap constraint gradually rather than silently falling back to top-N.
- Per-fund weight floor and cap applied after score-proportional allocation.

## 1. Environment Setup

In [1]:
%load_ext autoreload
%autoreload 2

import logging
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations

np.random.seed(42)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

logging.basicConfig(level=logging.INFO, format='%(levelname)s — %(message)s')
logger = logging.getLogger(__name__)

## 2. Paths & Parameters

Edit this cell only — no need to touch logic cells.

In [2]:
# ── Paths ──────────────────────────────────────────────────────────────────────
RAW_FUNDS_PATH   = Path('../mutualfunds/raw_funds.tsv')
FUND_INFO_DIR    = Path('../mutualfunds/fund_info')
FUND_SCORES_PATH = Path('../mutualfunds/fund_scores.tsv')  # precomputed by fund_selection_strategy.ipynb
OUTPUT_DIR       = Path('tuning_results/portfolio_debug')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Portfolio Parameters ───────────────────────────────────────────────────────
NUM_FUNDS    = 8
RISK_PROFILE = 'aggressive'   # conservative | moderate | aggressive

# ── Hybrid Score Weights ───────────────────────────────────────────────────────
# abs_score  : precomputed upstream (10Y/5Y/3Y percentile + stability + Crisil)
# risk_score : peer-relative, computed below
# Both normalized to [0, 100] within universe before blending.
W_ABS  = 0.50
W_RISK = 0.50

# ── Risk Score Component Weights (must sum to 1.0) ────────────────────────────
# Sharpe 3Y  : risk-adjusted return, recent cycle
# Sortino 3Y : downside risk, recent cycle
# Sharpe 5Y  : medium-term consistency — counters 3Y procyclicality
# Expense ratio : cost drag vs category peers (inverted: cheaper = better)
# AUM        : capacity/liquidity drag (small & mid cap only; flexi/L&M exempt)
W_SHARPE_3Y = 0.25
W_SORTINO   = 0.20
W_SHARPE_5Y = 0.25
W_EXPENSE   = 0.20
W_AUM       = 0.10

assert abs(W_SHARPE_3Y + W_SORTINO + W_SHARPE_5Y + W_EXPENSE + W_AUM - 1.0) < 1e-9, \
    'Risk score component weights must sum to 1.0'

# ── AUM Penalty Parameters ─────────────────────────────────────────────────────
# Applies only to small cap and mid cap funds (by name match).
# No penalty below AUM_LOW. Full penalty above AUM_HIGH. Linear in between.
# Flexi cap / large & mid cap exempt — mandate allows large cap rotation.
AUM_LOW  = 5_000    # Cr — no penalty at or below this
AUM_HIGH = 12_000   # Cr — full penalty at or above this
AUM_PENALTY_CATEGORIES = ('small cap', 'mid cap', 'midcap')  # case-insensitive

# ── Weight Allocation Bounds ───────────────────────────────────────────────────
# Clipped and renormalized after score-proportional allocation.
# For NUM_FUNDS=8: floor=6.25%, cap=31.25%
WEIGHT_FLOOR = 1.0 / (2.0 * NUM_FUNDS)
WEIGHT_CAP   = 2.5 / NUM_FUNDS

# ── Overlap Constraint ─────────────────────────────────────────────────────────
MAX_OVERLAP_PCT = 40.0

# ── Fund Universe ──────────────────────────────────────────────────────────────
FUND_NAMES = [
    'HDFC Mid Cap Dir Gr',
    'Edelweiss Mid Cap Dir Gr',
    'Nippon India Growth Mid Cap Dir Gr',
    'Invesco India Mid Cap Dir Gr',
    'ICICI Pru MidCap Dir Gr',
    'Nippon India Small Cap Dir Gr',
    'Quant Small Cap Dir Gr',
    'HDFC Small Cap Dir Gr',
    'Bandhan Small Cap Dir Gr',
    'HSBC Value Dir Gr',
    'Quant Flexi Cap Dir Gr',
    'HDFC Flexi Cap Dir Gr',
    'Parag Parikh Flexi Cap Dir Gr',
    'Bandhan Large & Mid Cap Dir Gr',
    'Motilal Oswal Large & Midcap Dir Gr',
    'ICICI Pru Large & Mid Cap Dir Gr'
]

# ── Validate prerequisites ─────────────────────────────────────────────────────
assert RAW_FUNDS_PATH.exists(),   f'raw_funds.tsv not found at {RAW_FUNDS_PATH}'
assert FUND_INFO_DIR.exists(),    f'fund_info dir not found at {FUND_INFO_DIR}'
assert FUND_SCORES_PATH.exists(), (
    f'fund_scores.tsv not found at {FUND_SCORES_PATH}. '
    'Run fund_selection_strategy.ipynb first.'
)
assert RISK_PROFILE in ('conservative', 'moderate', 'aggressive'), \
    f'Invalid RISK_PROFILE: "{RISK_PROFILE}". Must be conservative | moderate | aggressive.'

logger.info(
    f'Parameters OK — {len(FUND_NAMES)} funds | profile={RISK_PROFILE} | '
    f'n={NUM_FUNDS} | max_overlap={MAX_OVERLAP_PCT}% | '
    f'weight=[{WEIGHT_FLOOR*100:.1f}%, {WEIGHT_CAP*100:.1f}%]'
)

INFO — Parameters OK — 16 funds | profile=aggressive | n=8 | max_overlap=40.0% | weight=[6.2%, 31.2%]


## 3. Utility Functions

In [3]:
def safe_float(x):
    """
    Convert x to float, stripping commas. Returns None on failure.
    Single definition used everywhere — no local overrides anywhere in this notebook.
    Callers must handle None explicitly; never silently default to 0.0.
    """
    try:
        return float(str(x).replace(',', ''))
    except (ValueError, TypeError):
        return None


def normalize_series(s: pd.Series) -> pd.Series:
    """
    Min-max normalize a Series to [0, 100].
    Returns 50.0 for all if all values are identical (neutral midpoint).
    """
    mn, mx = s.min(), s.max()
    if mx == mn:
        return pd.Series(50.0, index=s.index)
    return (s - mn) / (mx - mn) * 100.0


def load_risk_row(isin: str, fund_info_dir: Path):
    """
    Load first row of risk_metrics_{isin}.tsv.
    Returns the row as a Series, or None if file missing/empty.
    """
    f = fund_info_dir / f'risk_metrics_{isin}.tsv'
    if not f.exists():
        return None
    df = pd.read_csv(f, sep='\t')
    return df.iloc[0] if not df.empty else None


def load_fund_scores(fund_scores_path: Path) -> dict:
    """
    Load precomputed absolute scores from fund_selection_strategy.ipynb.
    Returns dict keyed by ISIN.
    """
    df = pd.read_csv(fund_scores_path, sep='\t')
    return df.set_index('isin').to_dict(orient='index')


def clip_and_renormalize_weights(weights: dict,
                                  floor: float,
                                  cap: float,
                                  max_iter: int = 50) -> dict:
    """
    Iteratively clip weights to [floor, cap] and renormalize until stable.
    Ensures no fund gets a negligible or dominant allocation.
    Converges in a small number of iterations for typical portfolio sizes.
    """
    n = len(weights)
    assert floor * n <= 1.0, f'Floor {floor:.3f} × {n} funds > 1.0 — infeasible.'
    assert cap  * n >= 1.0, f'Cap {cap:.3f} × {n} funds < 1.0 — infeasible.'

    w = dict(weights)
    for _ in range(max_iter):
        total   = sum(w.values())
        w       = {k: v / total for k, v in w.items()}
        clipped = {k: min(max(v, floor), cap) for k, v in w.items()}
        if clipped == w:
            break
        w = clipped

    total = sum(w.values())
    return {k: v / total for k, v in w.items()}


logger.info('Utility functions defined.')

INFO — Utility functions defined.


## 4. Core Logic

### 4.1 Data Screen

No fund is dropped. Logs metric coverage so data gaps are visible before scoring.

In [4]:
def screen_funds(fund_isins: list, fund_info_dir: Path) -> pd.DataFrame:
    """
    Soft data screen — logs coverage, drops nothing.

    For each fund, checks availability of every metric used in scoring.
    Funds with partial data proceed; missing components score 50 pts (neutral).
    Returns a DataFrame with one row per fund and boolean coverage columns
    so the analyst can spot gaps before trusting the output.
    """
    REQUIRED_FIELDS = [
        'sharpe_3y', 'sharpe_cat_avg_3y',
        'sortino_3y', 'sortino_cat_avg_3y',
        'sharpe_5y', 'sharpe_cat_avg_5y',
        'expense_ratio', 'expense_ratio_cat_avg',
        'aum',
    ]

    rows = []
    for isin in fund_isins:
        row = load_risk_row(isin, fund_info_dir)
        entry = {'ISIN': isin, 'file_found': row is not None}
        for field in REQUIRED_FIELDS:
            entry[field] = (row is not None) and (safe_float(row.get(field)) is not None)
        rows.append(entry)

    df = pd.DataFrame(rows)

    no_file = df[~df['file_found']]['ISIN'].tolist()
    if no_file:
        logger.warning(f'No risk_metrics file for {len(no_file)} funds: {no_file}')

    for field in REQUIRED_FIELDS:
        gap = df[~df[field]]['ISIN'].tolist()
        if gap:
            logger.warning(f'  [{field}] missing for {len(gap)} funds — will score neutral (50): {gap}')

    full = df[REQUIRED_FIELDS].all(axis=1).sum()
    logger.info(
        f'Data screen: {len(fund_isins)} funds total | '
        f'{full} full coverage | {len(fund_isins)-full} partial (neutral on gaps)'
    )
    return df


logger.info('screen_funds defined.')

INFO — screen_funds defined.


### 4.2 Ranking

In [5]:
def _aum_score(aum_cr, fund_name: str) -> float:
    """
    AUM capacity score for small and mid cap funds only.

    Large AUM in small/mid cap is a documented performance drag: the fund
    cannot take meaningful positions in genuinely small companies without
    moving the price against itself. Flexi cap and large & mid cap are
    exempt because their mandate allows large cap rotation as AUM grows.

    Returns:
        100  if fund is not in a penalised category, or AUM <= AUM_LOW
        0    if AUM >= AUM_HIGH
        Linear interpolation between AUM_LOW and AUM_HIGH
    """
    is_penalised = any(cat in fund_name.lower() for cat in AUM_PENALTY_CATEGORIES)
    if not is_penalised or aum_cr is None:
        return 100.0
    if aum_cr <= AUM_LOW:
        return 100.0
    if aum_cr >= AUM_HIGH:
        return 0.0
    return 100.0 * (AUM_HIGH - aum_cr) / (AUM_HIGH - AUM_LOW)


def rank_funds(fund_isins: list,
               fund_info_dir: Path,
               fund_scores_path: Path,
               isin_to_name: dict,
               w_abs:  float = W_ABS,
               w_risk: float = W_RISK) -> pd.DataFrame:
    """
    Hybrid scoring: absolute quality (w_abs) + peer-relative risk (w_risk).

    abs_score (0-100):
        Precomputed upstream. Encodes long-term return percentile (10Y/5Y/3Y),
        stability CV, track-record tier, Crisil rating. Treated as ground
        truth — not recomputed here. Missing ISIN falls back to 50 (neutral).

    risk_score (0-100) — component weights from parameters cell:
        Sharpe 3Y  (W_SHARPE_3Y) : risk-adjusted return, recent cycle
        Sortino 3Y (W_SORTINO)   : downside risk only, recent cycle
        Sharpe 5Y  (W_SHARPE_5Y) : medium-term consistency; counters 3Y procyclicality
        Expense ratio (W_EXPENSE): cost drag vs category average (inverted — cheaper is better)
        AUM (W_AUM)              : capacity drag for small/mid cap funds

    Ratio-based components scaled as: fund/category_avg * 50
        0.0x avg → 0 pts | 1.0x avg → 50 pts | 2.0x avg → 100 pts
    Expense ratio: inverted (category_avg/fund * 50) — cheaper than avg > 50
    Missing metric → 50 pts (neutral) for that component only.

    Blending:
        abs_score and risk_score normalized to [0,100] within the universe
        before blending, making the w_abs/w_risk split genuinely proportional.
    """
    abs_lookup = load_fund_scores(fund_scores_path)
    rows = []

    for isin in fund_isins:
        row       = load_risk_row(isin, fund_info_dir)
        fund_name = isin_to_name.get(isin, '')

        def get(field):
            return safe_float(row.get(field)) if row is not None else None

        def ratio_score(val, avg, invert=False):
            """Ratio vs category avg → [0, 100]. None inputs → 50 (neutral)."""
            if val is None or avg is None or avg == 0:
                return 50.0
            r = (avg / val) if invert else (val / avg)
            return min(100.0, max(0.0, r * 50.0))

        s3y_score  = ratio_score(get('sharpe_3y'),     get('sharpe_cat_avg_3y'))
        sort_score = ratio_score(get('sortino_3y'),    get('sortino_cat_avg_3y'))
        s5y_score  = ratio_score(get('sharpe_5y'),     get('sharpe_cat_avg_5y'))
        er_score   = ratio_score(get('expense_ratio'), get('expense_ratio_cat_avg'), invert=True)
        aum_sc     = _aum_score(get('aum'), fund_name)

        risk_score = (
            W_SHARPE_3Y * s3y_score  +
            W_SORTINO   * sort_score +
            W_SHARPE_5Y * s5y_score  +
            W_EXPENSE   * er_score   +
            W_AUM       * aum_sc
        )

        info      = abs_lookup.get(isin, {})
        abs_score = info.get('abs_score', 50.0)

        rows.append({
            'ISIN':             isin,
            'abs_score':        round(abs_score,  2),
            'risk_score':       round(risk_score, 2),
            # component breakdown
            'sharpe_3y_score':  round(s3y_score,  2),
            'sortino_score':    round(sort_score,  2),
            'sharpe_5y_score':  round(s5y_score,  2),
            'expense_score':    round(er_score,    2),
            'aum_score':        round(aum_sc,      2),
            # raw values
            'sharpe_3y':        get('sharpe_3y'),
            'sortino_3y':       get('sortino_3y'),
            'sharpe_5y':        get('sharpe_5y'),
            'expense_ratio':    get('expense_ratio'),
            'aum_cr':           get('aum'),
            # abs metadata
            'tier':   info.get('tier', '?'),
            'CV':     info.get('CV'),
            's10Y':   info.get('s10Y'),
            's5Y':    info.get('s5Y'),
            's_stab': info.get('s_stab'),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    # Normalize within universe so blend is genuinely 50/50
    df['abs_norm']  = normalize_series(df['abs_score'])
    df['risk_norm'] = normalize_series(df['risk_score'])
    df['score']     = (w_abs * df['abs_norm'] + w_risk * df['risk_norm']).round(2)

    return df.sort_values('score', ascending=False).reset_index(drop=True)


logger.info('rank_funds defined.')

INFO — rank_funds defined.


### 4.3 Overlap Matrix

In [6]:
def compute_overlap_matrix(ranked_df: pd.DataFrame, fund_info_dir: Path) -> pd.DataFrame:
    """
    Pairwise holding overlap between funds (%).

    Overlap(A, B) = sum(min(w_a, w_b)) / avg(total_A, total_B) * 100

    Denominator is the AVERAGE of both totals (Sørensen-style).
    The previous min() denominator biased overlap upward for small funds
    paired with large ones, unfairly penalising large-small combinations.

    Funds with missing holdings score 0% overlap (conservative: not penalised
    for missing data, but also contribute no diversification evidence).
    """
    isins    = ranked_df['ISIN'].tolist()
    holdings = {}
    missing  = []

    for isin in isins:
        f = fund_info_dir / f'holdings_{isin}.tsv'
        if not f.exists():
            holdings[isin] = {}
            missing.append(isin)
            continue
        df = pd.read_csv(f, sep='\t')
        if df.empty:
            holdings[isin] = {}
            missing.append(isin)
            continue
        df['weight'] = df['weight'].apply(safe_float).fillna(0.0)
        holdings[isin] = df.groupby('stock_name')['weight'].sum().to_dict()

    if missing:
        logger.warning(
            f'Holdings missing for {len(missing)} funds '
            f'(overlap treated as 0%): {missing}'
        )

    n       = len(isins)
    overlap = pd.DataFrame(0.0, index=isins, columns=isins)

    for i in range(n):
        for j in range(i, n):
            a, b = isins[i], isins[j]
            if i == j:
                overlap.loc[a, b] = 100.0
                continue
            stocks_a, stocks_b = holdings[a], holdings[b]
            all_stocks = set(stocks_a) | set(stocks_b)
            raw   = sum(min(stocks_a.get(s, 0.0), stocks_b.get(s, 0.0)) for s in all_stocks)
            denom = (sum(stocks_a.values()) + sum(stocks_b.values())) / 2.0
            pct   = (raw / denom * 100.0) if denom > 0 else 0.0
            overlap.loc[a, b] = pct
            overlap.loc[b, a] = pct

    return overlap


logger.info('compute_overlap_matrix defined.')

INFO — compute_overlap_matrix defined.


### 4.4 Portfolio Optimizer

In [7]:
def optimize_portfolio(ranked_df:      pd.DataFrame,
                       overlap_matrix:  pd.DataFrame,
                       num_funds:       int   = NUM_FUNDS,
                       max_overlap:     float = MAX_OVERLAP_PCT,
                       weight_floor:    float = WEIGHT_FLOOR,
                       weight_cap:      float = WEIGHT_CAP) -> dict:
    """
    Combination-based portfolio optimizer.

    Selection:
        Evaluates all C(n, num_funds) combinations.
        Rejects any where any pairwise overlap > max_overlap.
        Selects the combo with the highest total hybrid score.

    Overlap fallback:
        If no combo passes, threshold relaxes in 5% steps up to 2x max_overlap,
        logged at each step. Raises ValueError if still no valid portfolio found.
        Replaces the previous silent top-N fallback.

    Weight allocation:
        Proportional to hybrid score — same objective as selection.
        Iteratively clipped to [weight_floor, weight_cap] and renormalized.
        No fund gets a negligible (<floor) or dominant (>cap) allocation.
    """
    isins     = ranked_df['ISIN'].tolist()
    score_map = dict(zip(ranked_df['ISIN'], ranked_df['score']))
    k         = min(num_funds, len(isins))

    def _find_best(threshold):
        best_combo, best_score = None, -float('inf')
        for combo in combinations(isins, k):
            if any(
                overlap_matrix.loc[combo[i], combo[j]] > threshold
                for i in range(len(combo))
                for j in range(i + 1, len(combo))
            ):
                continue
            total = sum(score_map[f] for f in combo)
            if total > best_score:
                best_score, best_combo = total, combo
        return best_combo, best_score

    best_combo, best_score = _find_best(max_overlap)
    effective_threshold    = max_overlap

    if best_combo is None:
        for relaxed in range(int(max_overlap) + 5, int(max_overlap * 2) + 1, 5):
            logger.warning(
                f'No valid portfolio at overlap ≤ {effective_threshold:.0f}%. '
                f'Relaxing to {relaxed}%.'
            )
            best_combo, best_score = _find_best(relaxed)
            effective_threshold    = relaxed
            if best_combo is not None:
                break

    if best_combo is None:
        raise ValueError(
            f'No valid portfolio found even at {effective_threshold:.0f}% overlap. '
            'Expand the fund universe or raise MAX_OVERLAP_PCT.'
        )

    raw   = {isin: max(score_map.get(isin, 0.0), 0.0) for isin in best_combo}
    total = sum(raw.values())
    initial = (
        {k: v / total for k, v in raw.items()}
        if total > 0
        else {isin: 1.0 / len(best_combo) for isin in best_combo}
    )
    weights = clip_and_renormalize_weights(initial, weight_floor, weight_cap)

    return {
        'selected_funds':        list(best_combo),
        'weights':               weights,
        'portfolio_score':       best_score,
        'effective_overlap_pct': effective_threshold,
    }


logger.info('optimize_portfolio defined.')

INFO — optimize_portfolio defined.


## 5. Load & Validate Data

In [8]:
raw_df = pd.read_csv(RAW_FUNDS_PATH, sep='\t')
raw_df.columns = [c.strip() for c in raw_df.columns]

assert 'schemeName' in raw_df.columns, 'Missing schemeName column in raw_funds.tsv'
assert 'isin'       in raw_df.columns, 'Missing isin column in raw_funds.tsv'

ISIN_TO_NAME = dict(zip(raw_df['isin'], raw_df['schemeName']))

def get_fund_name(isin: str) -> str:
    return ISIN_TO_NAME.get(isin, f'Unknown ({isin})')

logger.info(f'Loaded {len(raw_df)} records from raw_funds.tsv')

INFO — Loaded 924 records from raw_funds.tsv


## 6. Resolve Fund Names → ISINs

In [9]:
def resolve_fund_names(names: list, raw_df: pd.DataFrame) -> pd.DataFrame:
    """Exact case-insensitive match of fund names to ISINs."""
    rows = []
    for name in names:
        match = raw_df[raw_df['schemeName'].str.lower() == name.lower()]
        isin  = str(match.iloc[0]['isin']) if not match.empty else None
        rows.append({'fund_name': name, 'isin': isin, 'resolved': isin is not None})
    return pd.DataFrame(rows)


resolve_df = resolve_fund_names(FUND_NAMES, raw_df)
unresolved = resolve_df[~resolve_df['resolved']]

if not unresolved.empty:
    logger.warning(f'{len(unresolved)} fund names could not be resolved to ISINs:')
    print(unresolved[['fund_name']].to_string(index=False))

fund_isins = resolve_df.loc[resolve_df['resolved'], 'isin'].tolist()
assert fund_isins, 'No funds resolved. Check FUND_NAMES against raw_funds.tsv.'

logger.info(f'Resolved {len(fund_isins)}/{len(FUND_NAMES)} fund names to ISINs')

INFO — Resolved 16/16 fund names to ISINs


## 7. Run Pipeline

In [10]:
# ── Step 1: Data Screen ───────────────────────────────────────────────────────
print('=' * 70)
print('STEP 1 — Data Screen (no funds dropped)')
print('=' * 70)

screen_df = screen_funds(fund_isins, FUND_INFO_DIR)
print(screen_df.to_string(index=False))

WARNING —   [expense_ratio] missing for 16 funds — will score neutral (50): ['INF179K01XQ0', 'INF843K01AO4', 'INF204K01E54', 'INF205K01MV6', 'INF109K011N7', 'INF204K01K15', 'INF966L01689', 'INF179KA1RW5', 'INF194KB1AL4', 'INF917K01HD4', 'INF966L01911', 'INF179K01UT0', 'INF879O01027', 'INF194K01V89', 'INF247L01999', 'INF109K011O5']
WARNING —   [expense_ratio_cat_avg] missing for 16 funds — will score neutral (50): ['INF179K01XQ0', 'INF843K01AO4', 'INF204K01E54', 'INF205K01MV6', 'INF109K011N7', 'INF204K01K15', 'INF966L01689', 'INF179KA1RW5', 'INF194KB1AL4', 'INF917K01HD4', 'INF966L01911', 'INF179K01UT0', 'INF879O01027', 'INF194K01V89', 'INF247L01999', 'INF109K011O5']
WARNING —   [aum] missing for 16 funds — will score neutral (50): ['INF179K01XQ0', 'INF843K01AO4', 'INF204K01E54', 'INF205K01MV6', 'INF109K011N7', 'INF204K01K15', 'INF966L01689', 'INF179KA1RW5', 'INF194KB1AL4', 'INF917K01HD4', 'INF966L01911', 'INF179K01UT0', 'INF879O01027', 'INF194K01V89', 'INF247L01999', 'INF109K011O5']
INF

STEP 1 — Data Screen (no funds dropped)
        ISIN  file_found  sharpe_3y  sharpe_cat_avg_3y  sortino_3y  sortino_cat_avg_3y  sharpe_5y  sharpe_cat_avg_5y  expense_ratio  expense_ratio_cat_avg   aum
INF179K01XQ0        True       True               True        True                True       True               True          False                  False False
INF843K01AO4        True       True               True        True                True       True               True          False                  False False
INF204K01E54        True       True               True        True                True       True               True          False                  False False
INF205K01MV6        True       True               True        True                True       True               True          False                  False False
INF109K011N7        True       True               True        True                True       True               True          False                  False 

In [11]:
# ── Step 2: Rank Funds ────────────────────────────────────────────────────────
print('=' * 70)
print('STEP 2 — Ranking (hybrid = abs + risk)')
print('=' * 70)

ranked_df = rank_funds(
    fund_isins, FUND_INFO_DIR, FUND_SCORES_PATH, ISIN_TO_NAME,
    w_abs=W_ABS, w_risk=W_RISK
)
assert not ranked_df.empty, 'Ranking returned no results. Check risk_metrics files.'
ranked_df['name'] = ranked_df['ISIN'].map(get_fund_name)

print('\n── Hybrid score ─────────────────────────────────────────────────────')
print(ranked_df[[
    'ISIN', 'score', 'abs_norm', 'risk_norm', 'abs_score', 'risk_score', 'tier', 'name'
]].to_string(index=True))

print('\n── Risk score components ────────────────────────────────────────────')
print(ranked_df[[
    'ISIN',
    'sharpe_3y_score', 'sortino_score', 'sharpe_5y_score', 'expense_score', 'aum_score',
    'sharpe_3y', 'sortino_3y', 'sharpe_5y', 'expense_ratio', 'aum_cr',
    'name'
]].to_string(index=True))

print('\n── Absolute score metadata ──────────────────────────────────────────')
print(ranked_df[['ISIN', 'abs_score', 'tier', 'CV', 's10Y', 's5Y', 's_stab', 'name']].to_string(index=True))

STEP 2 — Ranking (hybrid = abs + risk)

── Hybrid score ─────────────────────────────────────────────────────
            ISIN  score    abs_norm   risk_norm  abs_score  risk_score tier                                 name
0   INF879O01027  88.39   76.786688  100.000000      95.11       85.13    A        Parag Parikh Flexi Cap Dir Gr
1   INF179K01UT0  87.11   75.150027   99.074074      94.51       84.87    A                HDFC Flexi Cap Dir Gr
2   INF917K01HD4  62.46  100.000000   24.928775     103.62       64.05    A                    HSBC Value Dir Gr
3   INF194K01V89  62.19   79.296236   45.085470      96.03       69.71    A       Bandhan Large & Mid Cap Dir Gr
4   INF179K01XQ0  53.86   62.493181   45.227920      89.87       69.75    A                  HDFC Mid Cap Dir Gr
5   INF843K01AO4  52.74   76.923077   28.561254      95.16       65.07    A             Edelweiss Mid Cap Dir Gr
6   INF204K01K15  51.94   83.578833   20.299145      97.60       62.75    A        Nippon India Sma

In [12]:
# ── Step 3: Overlap Matrix ────────────────────────────────────────────────────
print('=' * 70)
print('STEP 3 — Overlap Matrix (Sørensen normalization)')
print('=' * 70)

overlap_df    = compute_overlap_matrix(ranked_df, FUND_INFO_DIR)
name_map      = {isin: get_fund_name(isin)[:22] for isin in overlap_df.index}
overlap_named = overlap_df.rename(index=name_map, columns=name_map)
print(overlap_named.round(1).to_string())

isins_list = overlap_df.index.tolist()
high_pairs = [
    (get_fund_name(isins_list[i])[:30],
     get_fund_name(isins_list[j])[:30],
     round(overlap_df.iloc[i, j], 1))
    for i in range(len(isins_list))
    for j in range(i + 1, len(isins_list))
    if overlap_df.iloc[i, j] > MAX_OVERLAP_PCT
]
if high_pairs:
    print(f'\n⚠️  Pairs above {MAX_OVERLAP_PCT:.0f}%:')
    for a, b, pct in sorted(high_pairs, key=lambda x: -x[2]):
        print(f'  {pct:5.1f}%  {a}  ↔  {b}')
else:
    print(f'\n✓ No pairs exceed {MAX_OVERLAP_PCT:.0f}%.')

STEP 3 — Overlap Matrix (Sørensen normalization)
                        Parag Parikh Flexi Cap  HDFC Flexi Cap Dir Gr  HSBC Value Dir Gr  Bandhan Large & Mid Ca  HDFC Mid Cap Dir Gr  Edelweiss Mid Cap Dir   Nippon India Small Cap  ICICI Pru Large & Mid   Nippon India Growth Mi  Bandhan Small Cap Dir   Quant Flexi Cap Dir Gr  Invesco India Mid Cap   ICICI Pru MidCap Dir G  Motilal Oswal Large &   Quant Small Cap Dir Gr  HDFC Small Cap Dir Gr
Parag Parikh Flexi Cap                   100.0                   34.9               19.4                    17.8                  2.3                     2.9                     8.5                    18.7                     3.3                     1.4                     9.1                     0.2                     0.1                     0.4                     4.5                    2.0
HDFC Flexi Cap Dir Gr                     34.9                  100.0               20.2                    26.6                 12.9                     5.3

In [13]:
# ── Step 4: Portfolio Optimization ───────────────────────────────────────────
print('=' * 70)
print('STEP 4 — Optimization')
print('=' * 70)

portfolio = optimize_portfolio(
    ranked_df, overlap_df,
    num_funds    = NUM_FUNDS,
    max_overlap  = MAX_OVERLAP_PCT,
    weight_floor = WEIGHT_FLOOR,
    weight_cap   = WEIGHT_CAP,
)

if portfolio['effective_overlap_pct'] > MAX_OVERLAP_PCT:
    logger.warning(
        f"Overlap constraint relaxed to {portfolio['effective_overlap_pct']:.0f}% "
        f"(target {MAX_OVERLAP_PCT:.0f}%). Consider expanding the fund universe."
    )

STEP 4 — Optimization


## 8. Results

In [14]:
print('=' * 70)
print(f'FINAL PORTFOLIO  |  {RISK_PROFILE.upper()}  |  {NUM_FUNDS} FUNDS')
print('=' * 70)
print(f'Portfolio score    : {portfolio["portfolio_score"]:.2f}')
print(f'Overlap constraint : {portfolio["effective_overlap_pct"]:.0f}%')
print(f'Weight bounds      : [{WEIGHT_FLOOR*100:.1f}%, {WEIGHT_CAP*100:.1f}%]')
print()

header = f'{"Wt%":>6}  {"Tier":>4}  {"Hybrid":>6}  {"Abs":>5}  {"Risk":>5}  '\
         f'{"S3Y":>5}  {"So3Y":>5}  {"S5Y":>5}  {"ER":>5}  {"AUM":>5}  Fund'
print(header)
print('-' * len(header))

result_rows = []
for isin, wt in sorted(portfolio['weights'].items(), key=lambda x: -x[1]):
    r    = ranked_df[ranked_df['ISIN'] == isin].iloc[0]
    name = get_fund_name(isin)
    print(
        f"{wt*100:5.1f}%  "
        f"{str(r['tier']):>4}  "
        f"{r['score']:6.2f}  "
        f"{r['abs_score']:5.1f}  "
        f"{r['risk_score']:5.1f}  "
        f"{r['sharpe_3y_score']:5.1f}  "
        f"{r['sortino_score']:5.1f}  "
        f"{r['sharpe_5y_score']:5.1f}  "
        f"{r['expense_score']:5.1f}  "
        f"{r['aum_score']:5.1f}  "
        f"{name}"
    )
    result_rows.append({
        'isin':            isin,
        'fund_name':       name,
        'weight_pct':      round(wt * 100, 2),
        'tier':            r['tier'],
        'hybrid_score':    r['score'],
        'abs_score':       r['abs_score'],
        'risk_score':      r['risk_score'],
        'sharpe_3y_score': r['sharpe_3y_score'],
        'sortino_score':   r['sortino_score'],
        'sharpe_5y_score': r['sharpe_5y_score'],
        'expense_score':   r['expense_score'],
        'aum_score':       r['aum_score'],
        'sharpe_3y':       r['sharpe_3y'],
        'sortino_3y':      r['sortino_3y'],
        'sharpe_5y':       r['sharpe_5y'],
        'expense_ratio':   r['expense_ratio'],
        'aum_cr':          r['aum_cr'],
    })

total_wt = sum(portfolio['weights'].values())
print('-' * len(header))
print(f"{total_wt*100:5.1f}%  {'':>4}  {'':>6}  {'':>5}  {'':>5}  TOTAL")

# ── Save ──────────────────────────────────────────────────────────────────────
out_path = OUTPUT_DIR / 'portfolio.tsv'
(
    pd.DataFrame(result_rows)
    .sort_values('weight_pct', ascending=False)
    .reset_index(drop=True)
    .to_csv(out_path, sep='\t', index=False)
)
logger.info(f'Portfolio saved → {out_path}')
print(f'\n✓ Saved to {out_path}')

INFO — Portfolio saved → tuning_results/portfolio_debug/portfolio.tsv


FINAL PORTFOLIO  |  AGGRESSIVE  |  8 FUNDS
Portfolio score    : 509.88
Overlap constraint : 40%
Weight bounds      : [6.2%, 31.2%]

   Wt%  Tier  Hybrid    Abs   Risk    S3Y   So3Y    S5Y     ER    AUM  Fund
---------------------------------------------------------------------------
 17.3%     A   88.39   95.1   85.1   92.9  100.0   87.7   50.0  100.0  Parag Parikh Flexi Cap Dir Gr
 17.1%     A   87.11   94.5   84.9   91.1  100.0   88.4   50.0  100.0  HDFC Flexi Cap Dir Gr
 12.2%     A   62.46  103.6   64.0   63.3   60.8   64.3   50.0  100.0  HSBC Value Dir Gr
 12.2%     A   62.19   96.0   69.7   70.2   74.8   68.8   50.0  100.0  Bandhan Large & Mid Cap Dir Gr
 10.6%     A   53.86   89.9   69.8   68.8   73.9   71.0   50.0  100.0  HDFC Mid Cap Dir Gr
 10.3%     A   52.74   95.2   65.1   64.4   66.1   63.1   50.0  100.0  Edelweiss Mid Cap Dir Gr
 10.2%     A   51.94   97.6   62.8   59.1   60.7   63.4   50.0  100.0  Nippon India Small Cap Dir Gr
 10.0%     A   51.19   86.1   71.1   69.7  